[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_3_camera_model/08_distortion_undistortion.ipynb)

# Notebook 08 — Distortion and Undistortion
### 6D Pose Vision Workshop · Part 3: Camera Model

**Estimated time:** 40 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video notes 3 (*Camera Calibration Explained*), 4 (*OpenCV Python Camera Calibration*)

---

## Recommended Watch

> These are the same videos recommended for NB 07 — they cover distortion and undistortion as well. Skip if you already watched them.

| # | Title | Link | Duration |
|---|---|---|---|
| 1 | **Camera Calibration in less than 5 Minutes with OpenCV** | [▶ Watch](https://www.youtube.com/watch?v=_-BTKiamRTg) | ~5 min |
| 2 | **OpenCV Python Camera Calibration (Intrinsic, Extrinsic, Distortion)** | [▶ Watch](https://www.youtube.com/watch?v=H5qbRTikxI4) | ~20 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, json

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(5, 4)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Load calibration from previous notebook (or use defaults)
calib_path = '../assets/calibration/camera_calibration.npz'

if os.path.exists(calib_path):
    data = np.load(calib_path)
    K    = data['K']
    dist = data['dist']
    print(f"Loaded calibration from {calib_path}")
else:
    # Use plausible defaults if notebook 07 wasn't run
    K    = np.array([[720, 0, 320], [0, 720, 240], [0, 0, 1]], dtype=np.float64)
    dist = np.array([-0.3, 0.1, 0.001, 0.002, -0.05])
    print("Using default calibration values (run notebook 07 first for real values)")

print(f"K:\n{K}")
print(f"dist: {dist.flatten()}")

## Where you left off in NB07 — and what happens next

At the end of NB07 you ran `cv2.calibrateCamera` and got two things:

```python
K    = [[fx,  0, cx],   # intrinsic matrix — maps camera coords to pixels
        [ 0, fy, cy],
        [ 0,  0,  1]]

dist = [k1, k2, p1, p2, k3]   # distortion coefficients — how your lens bends light
```

The code cell above loads those results from the `.npz` file NB07 saved. If you see "Using default calibration values", go back and run NB07 first.

**What NB07 told us:** the camera has distortion. The `dist` values are non-zero.

**What NB08 does:** actually *fix* it — remove the distortion from images so that straight lines in the world look straight in the image.

---

### What's recap and what's new in this notebook

> **Section 1 — Types of Lens Distortion** → **recap from NB07**. You already saw radial and tangential distortion explained in NB07 Section 3. This section repeats it briefly with visualizations. Skim it to refresh the concept, or skip straight to Section 2 if it's still fresh.

> **Section 2 onwards → new content.** This is where you learn the actual tools: `cv2.undistort`, `cv2.remap`, `getOptimalNewCameraMatrix`, the `alpha` parameter, and the real-time undistortion loop. None of this was in NB07.

---

## Learning Objectives

By the end of this notebook you will:

- Understand what **radial** and **tangential** distortion are physically
- Know the **5 distortion coefficients** `[k1, k2, p1, p2, k3]` and what each affects
- Apply `cv2.undistort` for simple one-shot undistortion
- Use `cv2.remap` for **real-time** undistortion (precompute maps, then apply per frame)
- Know when to use `alpha=0` vs `alpha=1` in `getOptimalNewCameraMatrix`
- Validate undistortion results visually

---

## 1. Types of Lens Distortion *(recap from NB07)*

> This section reviews what you already learned in NB07 Section 3 — same concepts, now with interactive visualizations so you can see the effect. If the theory is still fresh, skim the diagrams and move on to Section 2.

### Why real cameras distort

The pinhole camera model assumes a perfect lens — straight rays, no bending. Real lenses bend light, and this bending increases the further you get from the optical axis (center). This creates **distortion**.

### Radial distortion — the most common type

Points are displaced radially outward or inward from the image center:

$$x_{\text{distorted}} = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$
$$y_{\text{distorted}} = y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$

Where $r^2 = x^2 + y^2$ is the squared distance from the image center.

**Annotation:**
- $k_1, k_2, k_3$ — radial distortion coefficients
- $r$ — normalized distance from center
- Negative $k_1$ → barrel distortion (pincushion if positive)

```
Barrel distortion (k1 < 0)       Pincushion distortion (k1 > 0)
  ┌───────────────┐               ┌──────────────────────────────┐
  │ ╔═══════════╗ │               │ ╔════╗                ╔════╗ │
  │ ║  bulges   ║ │               │ ║    ║                ║    ║ │
  │ ║  outward  ║ │               │ ║    ║    pinches     ║    ║ │
  │ ╚═══════════╝ │               │ ╚════╝    inward      ╚════╝ │
  └───────────────┘               └──────────────────────────────┘
  (wide-angle, fisheye lenses)    (telephoto, budget webcams)
```

### Tangential distortion — lens misalignment

Caused by the lens not being perfectly parallel to the image sensor:

$$x_{\text{distorted}} = x + [2p_1 xy + p_2(r^2 + 2x^2)]$$
$$y_{\text{distorted}} = y + [p_1(r^2 + 2y^2) + 2p_2 xy]$$

Where $p_1, p_2$ are the tangential distortion coefficients. Usually very small in modern cameras.

### OpenCV's distortion coefficient vector

$$\mathbf{d} = [k_1, \; k_2, \; p_1, \; p_2, \; k_3]$$

This is the `dist` array returned by `calibrateCamera`. For most cameras, $|k_3|$ and $|p_1|, |p_2|$ are very small.

In [ ]:
# Visualize what different distortion coefficients look like

def simulate_distortion_for_viz(image, k1, k2=0, k3=0, K_mat=None):
    """Simulate how a distorted camera would have captured this clean image.

    How the trick works:
      cv2.initUndistortRectifyMap produces the *undistort* sampling map —
      for each output pixel it says "go sample from this location in the
      distorted input image".  When we apply that map to a *clean* image
      instead of a distorted one, the output looks like what a real lens
      with these coefficients would have captured: straight grid lines get
      pulled into barrel or pincushion curves.

    This is a visualization approximation, not a true forward distortion
    model, but it is accurate enough to show the qualitative effect of
    each coefficient.
    """
    h, w = image.shape[:2]
    if K_mat is None:
        K_mat = np.array([[w*0.8, 0, w/2], [0, w*0.8, h/2], [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.array([k1, k2, 0, 0, k3])

    # Build the undistort sampling map and apply it to the clean image.
    # Result: the image looks as if it was captured by a lens with these
    # distortion coefficients.
    map1, map2 = cv2.initUndistortRectifyMap(K_mat, dist_coeffs, None, K_mat, (w, h), cv2.CV_32FC1)
    return cv2.remap(image, map1, map2, cv2.INTER_LINEAR)

# Create a grid image (straight lines show distortion clearly)
def create_grid_image(h=400, w=400, spacing=40):
    img = np.ones((h, w, 3), dtype=np.uint8) * 240
    for x in range(0, w, spacing):
        cv2.line(img, (x, 0), (x, h), (150, 150, 150), 1)
    for y in range(0, h, spacing):
        cv2.line(img, (0, y), (w, y), (150, 150, 150), 1)
    # Add a few colored reference points
    for x in range(spacing, w, spacing*2):
        for y in range(spacing, h, spacing*2):
            cv2.circle(img, (x, y), 4, (200, 50, 50), -1)
    return img

grid = create_grid_image()

K_vis = np.array([[320, 0, 200], [0, 320, 200], [0, 0, 1]], dtype=np.float64)

cases = [
    (grid,                                                             'Original (no distortion)'),
    (simulate_distortion_for_viz(grid, -0.5, 0.2, 0, K_vis), 'k1=-0.5 (barrel)'),
    (simulate_distortion_for_viz(grid,  0.5, 0.0, 0, K_vis), 'k1=+0.5 (pincushion)'),
    (simulate_distortion_for_viz(grid, -0.8, 0.5, 0, K_vis), 'k1=-0.8, k2=+0.5 (fisheye-like)'),
]

show_images(cases, figsize_per=(4, 4))
print("Grid lines should be straight — distortion bends them")

## 2. Undistortion Methods ← new content starts here

You have K and `dist` from NB07. Now the question is: how do you actually *remove* the distortion from an image?

This is where NB08 diverges from NB07. Everything below is new.

---

You have K and distortion coefficients from calibration. Now you need to actually remove the distortion from images. OpenCV gives you two approaches — they produce identical results but differ enormously in speed:

- **`cv2.undistort`** — single function call, simple. Internally recomputes the pixel mapping on every call. Use this for offline processing of individual images.
- **`cv2.remap`** — precompute the mapping once, then apply it to every frame cheaply. **3–5× faster per frame** than `undistort`. Use this for any real-time loop.

The right choice depends on your use case: processing a folder of images → `undistort`. Camera feed on a robot → `remap`.

### Method A: cv2.undistort (simple, best for single images)

```python
undistorted = cv2.undistort(img, K, dist, None, K_new)
```

Internally: recomputes the pixel mapping on every call. Fine for offline processing, slow for real-time.

### Method B: cv2.remap (fast, best for video)

Split into two steps:

```python
# Step 1: precompute maps ONCE before the video loop
mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)

# Step 2: apply maps to every frame (very fast)
undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
```

The maps are arrays of the same size as the image, storing for each output pixel where to sample from the distorted input. Computing them once and reusing is **3–5× faster** than `undistort` per frame.

### getOptimalNewCameraMatrix — controlling the crop

```python
K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
```

- `alpha=0` → crop to show only valid (undistorted) pixels — no black borders
- `alpha=1` → keep all pixels, including black border regions where distortion mapped outside
- `roi` → valid pixel region (use to crop if `alpha=0`)

In [ ]:
# Create a test image with strong distortion applied

# First, create a clean grid image
h, w = 480, 640
clean = create_grid_image(h, w, spacing=50)

# Apply strong barrel distortion (simulate a cheap wide-angle camera)
K_strong = np.array([[600, 0, w/2], [0, 600, h/2], [0, 0, 1]], dtype=np.float64)
dist_strong = np.array([-0.4, 0.15, 0.001, 0.002, -0.05])

# Distort the image (simulate what the camera captures)
map1, map2 = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_strong, (w, h), cv2.CV_32FC1)
distorted = cv2.remap(clean, map1, map2, cv2.INTER_LINEAR)

# --- Method A: cv2.undistort ---
K_new_a, roi_a = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=1)
undist_a = cv2.undistort(distorted, K_strong, dist_strong, None, K_new_a)

# --- Method B: cv2.remap (precomputed maps) ---
K_new_b, roi_b = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=0)
mapx_b, mapy_b = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_b, (w, h), cv2.CV_32FC1)
undist_b = cv2.remap(distorted, mapx_b, mapy_b, cv2.INTER_LINEAR)

# Show results
show_images([
    (clean,     'Original clean grid'),
    (distorted, 'Distorted (barrel k1=-0.4)'),
    (undist_a,  'undistort (alpha=1, full frame)'),
    (undist_b,  'remap (alpha=0, cropped to valid)'),
])

print(f"K_new (alpha=0): fx={K_new_b[0,0]:.1f}, cx={K_new_b[0,2]:.1f}")
print(f"Valid ROI (alpha=0): {roi_b}")

In [ ]:
# Compare alpha=0 vs alpha=1 behavior

results = []
for alpha in [0.0, 0.5, 1.0]:
    K_new_a, roi = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha)
    mapx_a, mapy_a = cv2.initUndistortRectifyMap(
        K_strong, dist_strong, None, K_new_a, (w, h), cv2.CV_32FC1)
    und = cv2.remap(distorted, mapx_a, mapy_a, cv2.INTER_LINEAR)
    results.append((und, f'alpha={alpha}\nroi={roi}'))

show_images(results)

print("alpha=0: all pixels valid, but image content is cropped")
print("alpha=1: full original field of view, black borders where distortion mapped outside")
print("\nFor robotics: use alpha=0 to avoid black border artifacts in pose estimation")

## 3. Real-Time Undistortion Loop

This is the pattern to use in all real-time robotics applications:

In [ ]:
# ── Real-time undistortion loop — reference pattern ──────────────────────────
#
# In a real script (not Jupyter) you would open a live camera:
#     cap = cv2.VideoCapture(0)
#     w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#
# Here we simulate it with synthetic frames so the cell runs to completion
# in a notebook. The structure — and the key rule — are identical:
#   PRECOMPUTE MAPS ONCE  →  APPLY remap EVERY FRAME
# ─────────────────────────────────────────────────────────────────────────────

import time

# ── 1. Load calibration (once, at startup) ───────────────────────────────────
K_rt    = K_strong.copy()     # in a real script: data = np.load('camera_calibration.npz')
dist_rt = dist_strong.copy()  #                   K_rt = data['K']; dist_rt = data['dist']
h_rt, w_rt = 480, 640

# ── 2. Precompute remap maps ONCE before the loop ────────────────────────────
K_new_rt, _ = cv2.getOptimalNewCameraMatrix(K_rt, dist_rt, (w_rt, h_rt), alpha=0)
mapx_rt, mapy_rt = cv2.initUndistortRectifyMap(
    K_rt, dist_rt, None, K_new_rt, (w_rt, h_rt), cv2.CV_32FC1)

print("Maps precomputed — K_new_rt ready for downstream pose estimation")
print(f"  K_new fx={K_new_rt[0,0]:.1f}  fy={K_new_rt[1,1]:.1f}  "
      f"cx={K_new_rt[0,2]:.1f}  cy={K_new_rt[1,2]:.1f}")

# ── 3. Per-frame loop ─────────────────────────────────────────────────────────
# In a real script:  while cap.isOpened(): ret, frame = cap.read()
# Here: 5 synthetic distorted frames to show the pattern runs at full speed.

N_FRAMES = 5
t0 = time.perf_counter()
for i in range(N_FRAMES):
    # Simulate what cap.read() returns: a distorted frame from the camera
    raw_frame  = cv2.remap(create_grid_image(h_rt, w_rt, 40),
                           mapx_rt, mapy_rt, cv2.INTER_LINEAR)

    # ← This one line is all the per-frame work:
    undistorted = cv2.remap(raw_frame, mapx_rt, mapy_rt, cv2.INTER_LINEAR)

    # All downstream work (ArUco detect, solvePnP, …) uses:
    #   frame    = undistorted   (distortion already removed)
    #   K        = K_new_rt      (updated intrinsics)
    #   dist     = np.zeros(5)   (no remaining distortion)

elapsed_ms = (time.perf_counter() - t0) / N_FRAMES * 1000
print(f"\nPer-frame remap: {elapsed_ms:.2f} ms  →  {1000/elapsed_ms:.0f} FPS ceiling")
print("\nKey rule: initUndistortRectifyMap BEFORE the loop, remap INSIDE the loop.")

## Running the real-time undistortion script

The script `scripts/pose/undistort_live_video.py` opens your webcam and shows a side-by-side comparison of the raw (distorted) frame and the corrected (undistorted) frame in real time. It precomputes the correction maps once at startup so the per-frame remapping is very fast.

---

### Before you run

You need your camera calibration file first. Run NB07 to generate `assets/calibration/camera_calibration.npz`. Without it the script uses a synthetic distortion fallback and the correction will not reflect your actual lens.

Then set up your virtual environment if you haven't already:

```bash
# 1. Create the venv (only once):
python -m venv venv

# 2. Activate it (every time you open a new terminal):
#    Windows:
venv\Scripts\activate
#    macOS / Linux:
source venv/bin/activate

# 3. Install dependencies (only once, after activating):
pip install -r requirements.txt
```

You should see `(venv)` at the start of your terminal prompt.  
If you don't, the venv is not active and the script will fail with `ModuleNotFoundError: No module named 'cv2'`.

---

### What the script does, step by step

1. Loads camera intrinsics (K, dist) from the calibration file.
2. Calls `getOptimalNewCameraMatrix` to compute a new K_new for the undistorted image — the `--alpha` parameter controls whether to crop black borders or keep the full field of view.
3. Calls `initUndistortRectifyMap` to build per-pixel remap lookup tables. This heavy step runs **once** at startup.
4. Each frame: applies `cv2.remap` using the precomputed tables — this is extremely fast and runs at full camera frame rate.
5. Labels each pane (`RAW` / `UNDISTORTED`) and overlays the current FPS.
6. In side-by-side mode, displays both half-size frames next to each other. Look at straight edges (door frames, window sills, walls) — they should appear curved in the left pane and straight in the right pane.

---

### How to run

```bash
# Default: auto-detects calibration, alpha=0 (no black borders)
python scripts/pose/undistort_live_video.py

# Specify calibration file explicitly
python scripts/pose/undistort_live_video.py --calib assets/calibration/camera_calibration.npz

# Keep full field of view (shows black borders at edges)
python scripts/pose/undistort_live_video.py --alpha 1
```

Key arguments:
- `--calib` — Path to the `.npz` calibration file from NB07.
- `--alpha` — `0` = crop to valid pixels only, no black borders (default). `1` = keep the full field of view, which shows black triangles at the corners.
- `--camera` — Camera index (default: `0`).
- `--width` / `--height` — Capture resolution (default: `1280 × 720`).

---

### Controls

| Key | Action |
|-----|--------|
| `Q` or `Esc` | Quit the live window |
| `S` | Toggle between side-by-side view and undistorted-only full-screen view |

---

### What output to expect

A window opens showing two half-size frames side by side, labelled `RAW` and `UNDISTORTED`. The FPS is shown in the top-right corner of the right pane.

At startup the terminal also prints the corrected camera matrix K_new:

```
Calibration loaded: assets/calibration/camera_calibration.npz
Camera     : 0  actual size 1280×720
alpha      : 0.0  (no black borders)
K_new      : fx=...  fy=...  cx=...  cy=...
```

No files are saved. If you use undistorted frames in a downstream pipeline (ArUco detection, solvePnP), use `K_new` as your camera matrix and pass `dist=zeros` — the distortion has already been removed by the remap step.

In [ ]:
# Benchmark: undistort vs remap performance

import time

# Create a test frame
test_frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Precompute remap maps
K_new_r, _ = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (640, 480), alpha=0)
mapx_r, mapy_r = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_r, (640, 480), cv2.CV_32FC1)

N = 100

# Benchmark cv2.undistort
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.undistort(test_frame, K_strong, dist_strong, None, K_new_r)
t_undistort = (time.perf_counter() - t0) / N * 1000

# Benchmark cv2.remap (precomputed)
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.remap(test_frame, mapx_r, mapy_r, cv2.INTER_LINEAR)
t_remap = (time.perf_counter() - t0) / N * 1000

print(f"cv2.undistort: {t_undistort:.2f} ms/frame → {1000/t_undistort:.0f} FPS max")
print(f"cv2.remap:     {t_remap:.2f} ms/frame  → {1000/t_remap:.0f} FPS max")
print(f"Speedup: {t_undistort/t_remap:.1f}×")
print("\n→ Use remap for video — precompute once, apply every frame")

## 4. Validating Undistortion

### The straight-line test

The easiest visual validation: straight physical lines (walls, doors, table edges) should appear straight after undistortion. If they're still curved, recalibrate.

In [ ]:
# Overlay reprojected calibration points — quantitative validation

# Take one calibration image, undistort it, and check that projected corners
# land where findChessboardCorners detects them

import glob

def load_calibration_npz(path):
    d = np.load(path)
    return d['K'], d['dist']

# Try to load from file, fallback to strong distortion example
K_val, dist_val = (K_strong, dist_strong)

# Create a grid image distorted by our model
orig = create_grid_image(480, 640, spacing=40)

# Add horizontal reference line (should be straight after undistortion)
cv2.line(orig, (0, 240), (640, 240), (0, 0, 200), 2)
cv2.line(orig, (320, 0), (320, 480), (0, 0, 200), 2)
cv2.putText(orig, 'These lines must be straight after undistort',
            (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 1)

# ── Distort step ──────────────────────────────────────────────────────────────
# simulate_distortion_for_viz applies the undistort map to a clean image;
# both steps must use the same output camera matrix (K_val) so the
# round-trip cancels cleanly.
map_d1, map_d2 = cv2.initUndistortRectifyMap(
    K_val, dist_val, None, K_val, (640, 480), cv2.CV_32FC1)
distorted_val = cv2.remap(orig, map_d1, map_d2, cv2.INTER_LINEAR)

# ── Undistort step ────────────────────────────────────────────────────────────
# Use K_val as the output matrix — same as the distort step above.
# If these two K matrices differ the round-trip will not cancel and the
# recovered lines will still appear slightly curved.
mapx_v, mapy_v = cv2.initUndistortRectifyMap(
    K_val, dist_val, None, K_val, (640, 480), cv2.CV_32FC1)
undistorted_val = cv2.remap(distorted_val, mapx_v, mapy_v, cv2.INTER_LINEAR)

show_images([
    (orig,            'Original (perfect grid)'),
    (distorted_val,   'After barrel distortion (lines curve)'),
    (undistorted_val, 'After undistortion (lines straight again)'),
])

print("If lines are straight again → undistortion is working correctly")
print("If lines are still curved → check that K and dist match the actual camera")

## Recap

| Concept | Key point |
|---|---|
| Radial distortion | $k_1, k_2, k_3$ — barrel (k1<0) or pincushion (k1>0) |
| Tangential distortion | $p_1, p_2$ — lens/sensor misalignment; usually small |
| dist vector | `[k1, k2, p1, p2, k3]` — OpenCV format |
| `undistort` | Simple; recomputes map every call; use for single images |
| `remap` | Fast; precompute with `initUndistortRectifyMap`; use for video |
| alpha=0 | Crop to valid pixels only (no black borders) |
| alpha=1 | Keep full FOV, include black border areas |
| After undistortion | Use `K_new` (not original K) and `dist=zeros` for pose estimation |

---

**Next:** `09_solvePnP_explained.ipynb` — now that we have K and can remove distortion, use corresponding 3D-2D point pairs to estimate the full 6D pose.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Effect of each distortion coefficient
# ============================================================
# Create a grid image and apply distortion with ONLY ONE coefficient nonzero:
#   1. k1 only (try -0.5 and +0.5)
#   2. k2 only (try +0.2)
#   3. p1 only (try +0.1)
#   4. p2 only (try +0.1)
# Display all 5 images and describe what each coefficient does.

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# grid = create_grid_image(400, 400, spacing=40)
# K_ex = np.array([[350, 0, 200], [0, 350, 200], [0, 0, 1]], dtype=np.float64)
# cases = []
# for name, d_arr in [
#     ('Original',    [0, 0, 0, 0, 0]),
#     ('k1=-0.5',     [-0.5, 0, 0, 0, 0]),
#     ('k1=+0.5',     [0.5, 0, 0, 0, 0]),
#     ('k2=+0.2',     [0, 0.2, 0, 0, 0]),
#     ('p1=+0.15',    [0, 0, 0.15, 0, 0]),
#     ('p2=+0.15',    [0, 0, 0, 0.15, 0]),
# ]:
#     d = np.array(d_arr)
#     if np.all(d == 0):
#         cases.append((grid, name))
#     else:
#         m1, m2 = cv2.initUndistortRectifyMap(K_ex, d, None, K_ex, (400,400), cv2.CV_32FC1)
#         cases.append((cv2.remap(grid, m1, m2, cv2.INTER_LINEAR), name))
#
# show_images(cases, figsize_per=(3, 3))
# # k1<0 → barrel (lines curve outward)
# # k1>0 → pincushion (lines curve inward)
# # k2 → adds higher-order radial correction
# # p1/p2 → diagonal shift (lines shift asymmetrically)

In [ ]:
# ============================================================
# EXERCISE 2: Build a complete undistortion function
# ============================================================
# Write undistort_frame(frame, K, dist, alpha=0) that:
#   1. Computes K_new with getOptimalNewCameraMatrix
#   2. Precomputes maps with initUndistortRectifyMap
#   3. Applies remap
#   4. Returns (undistorted_frame, K_new)
#
# Note: in a real video loop, precompute maps OUTSIDE the loop.
# This exercise is for the single-frame case.

# YOUR CODE HERE
def undistort_frame(frame, K, dist, alpha=0):
    pass


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def undistort_frame(frame, K, dist, alpha=0):
#     h, w = frame.shape[:2]
#     K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
#     mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)
#     undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
#     return undistorted, K_new
#
# # Test
# test_frame = distorted_val.copy()
# result, K_new_out = undistort_frame(test_frame, K_val, dist_val, alpha=0)
# print(f"Output shape: {result.shape}")
# print(f"K_new: fx={K_new_out[0,0]:.1f}, cx={K_new_out[0,2]:.1f}")
# show_images([(test_frame, 'Distorted'), (result, 'Undistorted')])